In [2]:
import os
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.llm import LLMChain
from langchain_classic.chains.combine_documents.stuff import StuffDocumentsChain
from langchain_core.prompts import PromptTemplate

In [3]:
# LOAD MS DOCUMENTS


loader = PyPDFLoader(r"C:\Python Data Analysis\LLM\AI Resume Critique\cv_computer-vision.pdf")
documents = loader.load()

documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-13T19:23:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-02-13T19:23:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Python Data Analysis\\LLM\\AI Resume Critique\\cv_computer-vision.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ALIFFA AGNUR\naliipa16@gmail.com +62 81242509833 linkedin.com/in/aliffaagnur\nhttps://kaggle.com/aliffaagnur https://github.com/BapakmuLah https://aliffaagnur.vercel.app/\nEDUCATION\nBachelor of Informatics Engineering Jul 2021–2025\nIndraprasta PGRI University - IPK 3.47/4.00\nEXPERIENCE\nMachine Learning Cohort - Bangkit Academy by Google July 2024–Jan 2025\n• Selected as one of 4636 accepted students from 45.841+ applicants across indonesia.\n• Designed and developed Machine Learning a

In [ ]:
# GET GEMINI API KEY

load_dotenv()
GOOGLE_API_KEY = os.getenv(key = 'GEMINI_API_KEY')

print(f'Connected to Gemini API : {GOOGLE_API_KEY[:5]}***')

In [4]:
# TEXT CHUNKING

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 100, separators = ["\n\n", "\n", " ", ""], keep_separator = True)

chunks = text_splitter.split_documents(documents= documents)
chunks[:5]

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-13T19:23:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-02-13T19:23:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Python Data Analysis\\LLM\\AI Resume Critique\\cv_computer-vision.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='ALIFFA AGNUR\naliipa16@gmail.com +62 81242509833 linkedin.com/in/aliffaagnur\nhttps://kaggle.com/aliffaagnur https://github.com/BapakmuLah https://aliffaagnur.vercel.app/\nEDUCATION'),
 Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-13T19:23:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-02-13T19:23:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 

In [6]:
# DEFINE VECTOR DB (FAISS)

# DEFINE EMBEDDING MODEL (GOOGLE EMBEDDING)
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")

# DEFINE VECTOR DATABASE (w FAISS)
vector_db = FAISS.from_documents(documents = chunks, embedding = embeddings)

vector_db

In [7]:
# SIMILARITY SEARCH

query = "can you give some insight based on my cv?"
retrieved_docs = vector_db.similarity_search(query)

retrieved_docs

[Document(id='c67eccfa-9624-4b88-bcd3-5406d2d88b91', metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-13T19:23:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-02-13T19:23:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Python Data Analysis\\LLM\\AI Resume Critique\\cv_computer-vision.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='2026\n• Completed advanced training in Machine Learning and Deep Learning, building and optimizing\npredictive models using modern neural network architectures.\nPROJECTS'),
 Document(id='c98b4a5e-9b47-44a0-bdf4-96e1d2b0274c', metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-13T19:23:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-02-13T19:23:12+00:00', 'ptex.fullbanner': 'This is pdfTeX

In [8]:
# CREATE PROMPT

PROMPT = """You are given the following documents: {docs}
            Based on this documents, answer the following question : {question}"""

In [9]:
# GENERATE RESULT WITH LLM

llm = ChatGoogleGenerativeAI(model = "models/gemma-3-27b-it")

prompt = PromptTemplate(template = PROMPT, input_variables = ["docs", "question"])
llm_chain = LLMChain(llm = llm, prompt = prompt)

llm_chain

C:\Users\aliff\AppData\Local\Temp\ipykernel_21584\793231376.py:6: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm = llm, prompt = prompt)


LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['docs', 'question'], input_types={}, partial_variables={}, template='You are given the following documents: {docs}\n            Based on this documents, answer the following question : {question}'), llm=ChatGoogleGenerativeAI(profile={}, google_api_key=SecretStr('**********'), model='models/gemma-3-27b-it', client=<google.genai.client.Client object at 0x0000025C51730650>, default_metadata=(), model_kwargs={}), output_parser=StrOutputParser(), llm_kwargs={})

In [10]:
rag_response = llm_chain.run(docs = retrieved_docs, question = query)

rag_response

C:\Users\aliff\AppData\Local\Temp\ipykernel_21584\2388453941.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  rag_response = llm_chain.run(docs = retrieved_docs, question = query)


'Okay, here\'s an insight based on the provided CV snippets:\n\n**Overall Impression:**\n\nThis CV appears to belong to a recent graduate (2025) or someone with very recent experience (early 2026) aiming for a role in Machine Learning/AI, specifically Computer Vision. The candidate has a strong academic background and has actively sought out relevant training opportunities.\n\n**Key Strengths:**\n\n*   **Strong Educational Foundation:** A Bachelor\'s in Informatics Engineering with a good GPA (3.47/4.00).\n*   **Focused Training:**  The candidate has completed advanced training in Machine Learning and Deep Learning, and participated in the Bangkit Academy by Google (a reputable program).  The AI Engineer Program Scholarship is also a positive.\n*   **Practical Experience:** Participation in the Bangkit Academy suggests hands-on project experience. The mention of projects involving classification models (Logistic Regression, SVM) and targeted marketing indicates application of skills.\n